# Solar Dataset — Exploratory Data Analysis (EDA)

**Important:** `solar.parquet` is treated as the **already-cleaned/final dataset**.

This notebook performs **EDA only**:
- no cleaning
- no imputation
- no row deletion
- no duplicate removal
- no feature engineering
- no data transformation for the analysis

The dataset is loaded with **pandas** using `pd.read_parquet()`.

## 1. Imports and project setup

The analysis uses pandas as the main data-analysis library, with NumPy and Matplotlib for supporting calculations and visualization.

In [ ]:
import sys
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

## 2. Load the cleaned Parquet dataset with pandas

`pd.read_parquet()` is intentionally used here. If pandas reports that no Parquet engine is installed, install one inside the active `.venv`:

```powershell
python -m pip install pyarrow
```

Then restart the kernel and rerun this cell.

In [ ]:
candidates = [
    pathlib.Path("data/raw/solar.parquet"),
    pathlib.Path("solar.parquet"),
    pathlib.Path("../data/raw/solar.parquet"),
    pathlib.Path("../solar.parquet"),
]

parquet_path = next((p for p in candidates if p.exists()), None)

if parquet_path is None:
    raise FileNotFoundError(
        "solar.parquet was not found. Expected it in data/raw/ or the project root."
    )

parquet_path = parquet_path.resolve()

# Pandas is the only library used to load the dataset.
df = pd.read_parquet(parquet_path)

print("File:", parquet_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
display(df.head())

## 3. Dataset overview

Understand the dataset structure before looking at individual variables.

In [ ]:
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nPandas info:")
df.info()

## 4. Missing values and duplicates — descriptive analysis only

Because this is cleaned data, this section **does not modify anything**. It only reports whether any missing values or duplicate rows remain in the supplied dataset.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100),
    "non_null_count": df.notna().sum(),
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

duplicate_count = int(df.duplicated().sum())
duplicate_percent = duplicate_count / max(len(df), 1) * 100

print(f"Duplicate rows present in cleaned dataset: {duplicate_count:,} ({duplicate_percent:.2f}%)")

## 5. Numeric variable summary

Use pandas descriptive statistics to understand center, spread, range, skew-related differences between mean and median, and extreme percentiles.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}):")
print(numeric_cols)

if numeric_cols:
    numeric_summary = df[numeric_cols].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T

    numeric_summary["median"] = df[numeric_cols].median()
    numeric_summary["skew"] = df[numeric_cols].skew()
    numeric_summary["missing"] = df[numeric_cols].isna().sum()

    display(numeric_summary)
else:
    print("No numeric columns detected.")

## 6. Categorical / object variable summary

Look at cardinality and the most common values without modifying the original data.

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Categorical/object columns ({len(categorical_cols)}):")
print(categorical_cols)

for col in categorical_cols:
    print(f"\n### {col}")
    print("Unique values:", df[col].nunique(dropna=True))
    display(
        df[col]
        .value_counts(dropna=False)
        .head(20)
        .rename("count")
        .to_frame()
    )

## 7. Numeric distributions

Histograms show the shape of each numeric variable and help identify skewness, heavy tails, concentration, and unusual ranges.

In [ ]:
if numeric_cols:
    for col in numeric_cols:
        s = df[col].dropna()

        plt.figure(figsize=(10, 4))
        plt.hist(s, bins=40)
        plt.title(f"Distribution — {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric columns to visualize.")

## 8. Boxplots and descriptive outlier inspection

Outliers are **reported, not removed**. In solar data, extreme observations can be meaningful operating conditions rather than errors.

In [ ]:
if numeric_cols:
    for col in numeric_cols:
        plt.figure(figsize=(10, 2.8))
        plt.boxplot(df[col].dropna(), vert=False)
        plt.title(f"Boxplot — {col}")
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        if pd.notna(iqr) and iqr != 0:
            count = int(((df[col] < lower) | (df[col] > upper)).sum())
            print(
                f"{col}: IQR range [{lower:.4g}, {upper:.4g}], "
                f"flagged observations = {count:,} ({count / max(len(df), 1) * 100:.2f}%)"
            )
else:
    print("No numeric columns to analyze.")

## 9. Correlation analysis

Correlation is exploratory and does **not** prove causation. Focus on strong relationships that may be useful for later analysis or modeling.

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()

    plt.figure(figsize=(max(8, 0.8 * len(numeric_cols)),
                        max(6, 0.7 * len(numeric_cols))))
    plt.imshow(corr, aspect="auto", interpolation="nearest")
    plt.colorbar(label="Pearson correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()

    # strongest unique pairs
    pairs = []
    for i, c1 in enumerate(corr.columns):
        for j, c2 in enumerate(corr.columns):
            if j <= i:
                continue
            value = corr.loc[c1, c2]
            if pd.notna(value):
                pairs.append((c1, c2, value, abs(value)))

    strongest = (
        pd.DataFrame(
            pairs,
            columns=["feature_1", "feature_2", "correlation", "abs_correlation"]
        )
        .sort_values("abs_correlation", ascending=False)
        .head(20)
    )

    display(strongest)
else:
    print("Need at least two numeric columns for correlation analysis.")

## 10. Datetime / time-series EDA

This section searches for columns that are already datetime typed or have date/time-like names. It does not alter the dataset.

In [ ]:
datetime_cols = list(df.select_dtypes(include=["datetime", "datetimetz"]).columns)

name_based_datetime_candidates = [
    c for c in df.columns
    if any(k in c.lower() for k in ["date", "time", "timestamp", "datetime"])
]

print("Datetime-typed columns:", datetime_cols)
print("Date/time-like column names:", name_based_datetime_candidates)

for col in datetime_cols:
    s = df[col].dropna()

    print(f"\n### {col}")
    print("Start:", s.min())
    print("End:", s.max())
    print("Unique timestamps:", s.nunique())

    if len(s) >= 2:
        diffs = s.sort_values().diff().dropna()
        print("Most common sampling intervals:")
        display(diffs.value_counts().head(10).rename("count").to_frame())

## 11. Solar-specific relationships

Where the column names make this possible, inspect relationships among common solar measurements such as irradiance, temperature, wind, humidity, and power/energy generation.

The code only visualizes columns that actually exist.

In [ ]:
def find_columns(keywords):
    return [
        c for c in df.columns
        if any(k in c.lower() for k in keywords)
    ]

solar_groups = {
    "irradiance": find_columns(["irradiance", "radiation", "solar_radiation", "ghi", "dni", "dhi"]),
    "power": find_columns(["power", "generation", "generated", "output"]),
    "energy": find_columns(["energy", "yield"]),
    "temperature": find_columns(["temp", "temperature"]),
    "humidity": find_columns(["humidity", "humid"]),
    "wind": find_columns(["wind", "windspeed", "wind_speed"]),
}

for group, cols in solar_groups.items():
    print(f"{group}: {cols}")

irr_cols = solar_groups["irradiance"]
power_cols = solar_groups["power"]

for x in irr_cols[:3]:
    for y in power_cols[:3]:
        tmp = df[[x, y]].dropna()

        if tmp.empty:
            continue

        plt.figure(figsize=(8, 5))
        plt.scatter(tmp[x], tmp[y], s=8, alpha=0.35)
        plt.xlabel(x)
        plt.ylabel(y)
        plt.title(f"{y} vs {x}")
        plt.tight_layout()
        plt.show()

        print(f"Pearson correlation: {tmp[x].corr(tmp[y]):.4f}")

## 12. Time-based solar trends

For a detected datetime column, this section plots available solar-related numeric variables against time. This helps reveal daily/seasonal patterns, trend changes, gaps, and volatility.

In [ ]:
if datetime_cols and numeric_cols:
    time_col = datetime_cols[0]

    temp = df[[time_col] + numeric_cols].copy()
    temp = temp.sort_values(time_col)

    for col in numeric_cols:
        plot_df = temp[[time_col, col]].dropna()

        if plot_df.empty:
            continue

        plt.figure(figsize=(13, 4))
        plt.plot(plot_df[time_col], plot_df[col], linewidth=0.7)
        plt.title(f"{col} over time")
        plt.xlabel(time_col)
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()
else:
    print("No datetime-typed column and numeric variables were found for time-series plots.")

## 13. Automated EDA findings table

This section creates a compact pandas table of useful descriptive signals. These are **observations from the data**, not cleaning decisions.

In [ ]:
findings = []

for col in numeric_cols:
    s = df[col].dropna()

    if s.empty:
        continue

    mean = s.mean()
    median = s.median()
    std = s.std()
    cv = std / mean if pd.notna(mean) and mean != 0 else np.nan

    findings.append({
        "column": col,
        "min": s.min(),
        "median": median,
        "mean": mean,
        "max": s.max(),
        "std": std,
        "skew": s.skew(),
        "coefficient_of_variation": cv,
        "unique": s.nunique(),
    })

findings_df = pd.DataFrame(findings)

if not findings_df.empty:
    display(
        findings_df.sort_values(
            "coefficient_of_variation",
            ascending=False,
            na_position="last"
        )
    )
else:
    print("No numeric findings available.")

## 14. EDA conclusions to write in the report

After running the notebook, summarize the actual results under these headings:

### Dataset structure
State the number of rows, columns, data types, and time coverage.

### Data quality observed
Because the dataset is already cleaned, simply report whether missing values or duplicates are present. Do not perform cleaning here.

### Distribution findings
Describe variables that are skewed, approximately symmetric, highly variable, or concentrated around particular ranges.

### Relationships
Discuss the strongest correlations and the most visible solar-specific relationships.

### Time patterns
Describe daily/seasonal patterns, trends, volatility, and sampling intervals where a datetime column exists.

### Outlier observations
Report variables with unusually high IQR-flagged values, while explicitly noting that these observations were not removed.

### Business / analytical implications
Explain what the observed distributions, relationships, and temporal patterns suggest for downstream solar analysis or predictive modeling.